## Loading the Abalone Dataset

The Abalone dataset contains physical measurements of abalone shells. Our goal is to predict the age, which is derived from the number of rings. We inspect the structure to understand the available features.

In [4]:
import pandas as pd
import numpy as np

col_names = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight", "Rings"
]
raw_df = pd.read_csv("abalone.data", header=None, names=col_names)

print("Number of rows:", len(raw_df))
print("Column names:", raw_df.columns.tolist())
print(raw_df.head())

Number of rows: 4177
Column names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


## Target Conversion and Feature Selection

Age is computed as Rings + 1.5. We use three numeric features — Length, Diameter, and Whole_weight — as inputs since they are strong physical indicators of abalone growth and maturity.

In [6]:
raw_df['age'] = raw_df['Rings'] + 1.5

selected_cols = ['Length', 'Diameter', 'Whole_weight']
X_all = raw_df[selected_cols].values
y_all = raw_df['age'].values.reshape(-1, 1)

## Train-Test Split and Normalization

We manually split 80% of data for training and 20% for testing. Feature normalization is applied using training statistics to keep gradient updates stable across features with different scales.

In [7]:
split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

train_mean = np.mean(X_train, axis=0)
train_std  = np.std(X_train, axis=0)

X_train = (X_train - train_mean) / train_std
X_test  = (X_test  - train_mean) / train_std

X_train shape: (3341, 3)
X_test shape: (836, 3)


## Model and Loss Definition

The linear model computes $\hat{y} = Xw + b$. We use Mean Squared Error as the loss — it penalizes larger errors more heavily due to the squaring term.

In [8]:
def linear_forward(X, weights, bias):
    return X @ weights + bias

def compute_mse(actual, predicted):
    return np.mean((actual - predicted) ** 2)

## Gradient Functions

We compute partial derivatives of MSE with respect to weights and bias. These gradients indicate how each parameter should change to reduce the loss.

In [9]:
def weight_gradient(X, actual, predicted):
    residual = predicted - actual
    return (2 / len(actual)) * (X.T @ residual)

def bias_gradient(actual, predicted):
    residual = predicted - actual
    return (2 / len(actual)) * np.sum(residual)

## Training Loop

Parameters are initialised with small random weights and zero bias. Each epoch runs a full forward pass, loss computation, gradient calculation, and parameter update.

In [10]:
weights = np.random.randn(X_train.shape[1], 1) * 0.01
bias    = 0.0
lr      = 0.1
n_epochs = 100

for ep in range(n_epochs):
    y_pred   = linear_forward(X_train, weights, bias)
    cur_loss = compute_mse(y_train, y_pred)
    dw       = weight_gradient(X_train, y_train, y_pred)
    db       = bias_gradient(y_train, y_pred)
    weights  = weights - lr * dw
    bias     = bias    - lr * db

    if ep % 10 == 0:
        print(f"Epoch {ep}, Loss: {cur_loss:.4f}")

Epoch 0, Loss: 144.2717
Epoch 10, Loss: 9.0104
Epoch 20, Loss: 7.4824
Epoch 30, Loss: 7.4557
Epoch 40, Loss: 7.4472
Epoch 50, Loss: 7.4396
Epoch 60, Loss: 7.4325
Epoch 70, Loss: 7.4260
Epoch 80, Loss: 7.4198
Epoch 90, Loss: 7.4141


## Evaluation on Test Set

We evaluate using MSE and MAE on the held-out test set and compare predicted vs actual age for the first 5 samples.

In [11]:
test_preds = linear_forward(X_test, weights, bias)
final_mse  = compute_mse(y_test, test_preds)
final_mae  = np.mean(np.abs(y_test - test_preds))

print(f"Test MSE: {final_mse:.4f}")
print(f"Test MAE: {final_mae:.4f}")

print("\nTrue Age | Predicted Age | Absolute Error")
for i in range(5):
    true_val = y_test[i][0]
    pred_val = test_preds[i][0]
    abs_err  = abs(true_val - pred_val)
    print(f"{true_val:8.2f} | {pred_val:13.2f} | {abs_err:14.2f}")

Test MSE: 5.3520
Test MAE: 1.8073

True Age | Predicted Age | Absolute Error
   13.50 |         11.06 |           2.44
   15.50 |          9.96 |           5.54
   14.50 |         10.50 |           4.00
   14.50 |         10.87 |           3.63
   13.50 |         10.91 |           2.59
